# 17 — SQLite history: step 1
Save and reload ConversationState without an LLM, GPU, model loading, or Agent changes.

This tests storage, not model understanding or full Agent resumption. The pending plan below is an explicit fixture. Store IDs and message ordering are preserved. The SQLite file contains the latest complete snapshot per conversation.


In [1]:
from pathlib import Path
import sys
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src/agent1").is_dir()), None)
if project_root is None:
    raise FileNotFoundError("Open this notebook inside your Agent1 project.")
sys.path.insert(0, str(project_root / "src"))
from agent1.persistence import SQLiteConversationStore
from agent1.conversation.state import ConversationState
from agent1.analysis.action_plan import ActionPlan
from agent1.conversation.context_builder import ContextBuilder
from uuid import uuid4

database_path = project_root / "data/local/conversations.sqlite3"
conversation_id = "notebook17-" + uuid4().hex
store = SQLiteConversationStore(database_path)
print("Database:", database_path)
print("Conversation:", conversation_id)


Database: d:\Projects\projects\agent1\data\local\conversations.sqlite3
Conversation: notebook17-054d570d373e491b8174b9beba908a44


## 1. Prepare a pending two-action conversation

In [2]:
state = ConversationState()
state.add_user_message("Track my order and check whether the blue backpack is available.")
state.add_assistant_message("Please provide your order number.")
plan = ActionPlan.model_validate({
    "request_type": "multi_action", "reason": "Storage test fixture",
    "steps": [
        {"intent": "track_order", "action": "order_service", "arguments": {}},
        {"intent": "availability", "action": "product_service",
         "arguments": {"product": "backpack", "color": "blue"}},
    ],
})
state.start_pending_action(
    plan, ["0.order_id", "1.product", "1.color"],
    {"1.product": "backpack", "1.color": "blue"},
)
print("Missing:", state.get_missing_arguments())
assert state.get_missing_arguments() == ["0.order_id"]


Missing: ['0.order_id']


## 2. Save, then read the database in a separate Python process
The subprocess proves that the snapshot is on disk rather than retained only in notebook memory. It does not import or load a model.

In [3]:
store.save(conversation_id, state)
import os, subprocess
child_code = """
import sys
from agent1.persistence import SQLiteConversationStore
s = SQLiteConversationStore(sys.argv[1]).load(sys.argv[2])
assert len(s.messages) == 2
assert len(s.pending_action_plan.steps) == 2
assert s.get_missing_arguments() == ['0.order_id']
assert s.pending_arguments['1.color'] == 'blue'
print('Separate-process restore: PASS')
"""
env = os.environ.copy()
env["PYTHONPATH"] = str(project_root / "src") + os.pathsep + env.get("PYTHONPATH", "")
check = subprocess.run([sys.executable, "-c", child_code, str(database_path), conversation_id],
                       env=env, capture_output=True, text=True)
print(check.stdout)
if check.returncode:
    print(check.stderr)
    raise RuntimeError("Separate-process restore failed.")


Separate-process restore: PASS



## 3. Reload and compare state and model context

In [4]:
restored = SQLiteConversationStore(database_path).load(conversation_id)
assert restored is not state
assert restored.messages == state.messages
assert restored.pending_action_plan == state.pending_action_plan
assert restored.pending_arguments == state.pending_arguments
assert restored.required_arguments == state.required_arguments
builder = ContextBuilder()
assert builder.build_for_analyzer(restored) == builder.build_for_analyzer(state)
print(builder.build_for_analyzer(restored))
print("Snapshot round trip: PASS")


user: Track my order and check whether the blue backpack is available.
assistant: Please provide your order number.
Snapshot round trip: PASS


## 4. Save a follow-up argument
This fills the stored state explicitly; it does not execute the tools. Agent integration comes next.

In [5]:
restored.add_user_message("ORD-123")
restored.update_pending_arguments(**{"0.order_id": "ORD-123"})
assert restored.get_missing_arguments() == []
store.save(conversation_id, restored)
reopened = SQLiteConversationStore(database_path).load(conversation_id)
assert reopened.pending_arguments["0.order_id"] == "ORD-123"
assert len(reopened.messages) == 3
assert state.get_missing_arguments() == ["0.order_id"]
assert store.load("missing-" + uuid4().hex) is None
print("Updated snapshot: PASS")
print("STEP 1 COMPLETE — no model calls were made.")


Updated snapshot: PASS
STEP 1 COMPLETE — no model calls were made.


## What is saved and what comes next
Messages (including metadata and tool messages), pending clarification/routing, the action plan, required arguments, and collected arguments are saved as JSON in SQLite.

The store is explicit: mutations after save require another save. Saving the same conversation ID replaces its row. No automatic Agent save/load is installed yet. Do not simply assign restored state to a live Agent: its runners hold state references and request text, and Agent also has a fresh-request flag. The next step will restore those together.

Keep `data/local/` out of Git by adding `/data/local/` to your root `.gitignore`. The database is local runtime data, not source code.
